In [1]:
import re
import os
import json
import sys
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict


In [42]:
print(os.getcwd())

C:\Users\egorg\Documents\RAG_минцифры


In [43]:
from docx import Document
import glob
path = "./Правильно"
path_WRONG = "./С ошибками"
docs_paths = glob.glob(os.path.join(path, "*.docx"))
docs_paths_WRONG = glob.glob(os.path.join(path_WRONG, "*.docx"))

contract_path_WRONG = docs_paths_WRONG[-1]
plan_path_CORRECT = docs_paths[0]
print(contract_path_WRONG)
print(plan_path_CORRECT)

./С ошибками\Проект Контракта.docx
./Правильно\1. Заявка в ПГ.docx


In [4]:
SPACE_CHARS = ["\u00a0", "\u202f", "\u2009", "\u2002", "\u2003", "\u2004", "\u2005", "\u3000", "\ufeff",
                "\xa0"]
QUOTES_MAP = {
    "«": '"', "»": '"',
    "“": '"', "”": '"', "„": '"', "‟": '"',
    "’": "'", "‘": "'",
}


In [44]:
# -------- ЗАЯВКА В ПЛАН ГРАФИК --------
class PlanParser:
    def __init__(self, path: str):
        self.path = path
        self.doc = Document(path)

    @staticmethod
    def normalize_text(text: str) -> str:
        """
        Нормализуем текст: пробелы, кавычки, тире, переносы
        """
        if text is None:
            return ""
        out = text
        out = out.replace("\n", "; ")
        for sp in SPACE_CHARS:
            out = out.replace(sp, " ")
        for k, v in QUOTES_MAP.items():
            out = out.replace(k, v)
        out = out.replace("–", "-").replace("—", "-").replace("−", "-")
        out = re.sub(r"[ \t\f\v]*\n[ \t\f\v]*", "\n", out)  # переносы оставляем как \n
        out = re.sub(r"[ \t]{2,}", " ", out)
        return out.strip()

    def extract_table_kv_from_docx(self) -> List[str]:
        """
        Извлекаем таблицу ключ-значение как список строк "ключ: значение"
        """
        kv = defaultdict(list)
        if not self.doc.tables:
            return []

        table = self.doc.tables[0]

        for row in table.rows:
            cells = [self.normalize_text(c.text) for c in row.cells]
            cells = cells[1:]  # пропускаем первый столбец (если нужно)
            if len(cells) < 2:
                continue
            key = self.normalize_text(cells[0])
            val = ", ".join(dict.fromkeys(cells[1:]))
            if key:
                kv[key].append(val)

        plain_text_lines = [f"{k}: {', '.join(vals)}" for k, vals in kv.items()]
        return plain_text_lines

    def extract_clean_text(self, chunk_size: int = 50) -> List[str]:
        """
        Извлекаем весь текст документа, делим на куски по chunk_size слов
        """
        full_text = []
        for para in self.doc.paragraphs:
            text = self.normalize_text(para.text)
            if text:
                full_text.append(text)

        # Разбиваем на куски по chunk_size слов
        chunks = []
        current_chunk = []
        for line in full_text:
            words = line.split()
            for w in words:
                current_chunk.append(w)
                if len(current_chunk) >= chunk_size:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = []
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

In [6]:
# -------- ПРОЕКТ КОНТРАКТА --------
class ContractParser:
    """ 
    Класс для извлечения и нормализации текста и таблиц из Word-документов.
    """

    def __init__(self, path: str):
        self.path = path
        self.doc = Document(path)

    @staticmethod
    def table_has_header(table) -> bool:
        """
        Проверка наличия заголовка у таблицы. Если есть, соединяю название колонки со значением в plain тексте
        """
        texts: List[str] = [c.text.strip() for c in table.rows[0].cells]
        had_full_digit_col: bool = any(all(ch.isdigit() for ch in t.replace('.', '')) for t in texts)
        #has_digits = any(any(ch.isdigit() for ch in t) for t in texts)
        return not had_full_digit_col and len(table.rows) > 1

    @staticmethod
    def normalize_text(text: str) -> str:
        """
        Вспомогательная функция нормализации текста
        """
        if text is None:
            return ""
        out = text
        for sp in SPACE_CHARS:
            out = out.replace(sp, " ")
        for k, v in QUOTES_MAP.items():
            out = out.replace(k, v)
        out = out.replace("–", "-").replace("—", "-").replace("−", "-")
        out = re.sub(r"[ \t\f\v]*\n[ \t\f\v]*", "\n", out)  # переносы оставляем как \n
        out = re.sub(r"[ \t]{2,}", " ", out)
        return out.strip()
    
    @staticmethod
    def chunk_text_words(text: str, chunk_size: int = 30, overlap: int = 10) -> List[str]:
        """
        Разбивает текст на куски по словам с пересечением
        Добавил, тк поиск ближайших векторов плохо работает с большими абзацами
        """
        words = text.split()
        if len(words) <= chunk_size:
            return [text]

        chunks = []
        start = 0
        while start < len(words):
            end = start + chunk_size
            chunk = " ".join(words[start:end])
            chunks.append(chunk)
            start += chunk_size - overlap
        return chunks
    
    def extract_clean_text(self, chunk_size: int = 30, overlap: int = 10) -> List[str]:
        """
        Извлчение всех абзацев
        """
        paragraphs: List[str] = []
        for p in self.doc.paragraphs:
            text: str = self.normalize_text(p.text)
            if not text:
                continue
            # разбиваем слишком длинные абзацы на чанки
            # Ну просто поиск по схожести плохо работает на больших
            chunks = self.chunk_text_words(text, chunk_size=chunk_size, overlap=overlap)
            paragraphs.extend(chunks)
        return paragraphs

    def extract_table_kv_from_docx(self) -> Dict[str, List[str]]:
        """
        Превращаю таблицы в plain-текст
        """
        doc = self.doc
        tables_kv: Dict[str, List[str]] = defaultdict(list)

        for i, table in enumerate(doc.tables):
            has_header: bool = self.table_has_header(table)
            if has_header:
                table_header: List[str] = [self.normalize_text(c.text) for c in table.rows[0].cells]

            for j, row in enumerate(table.rows):
                cells: List[str] = [self.normalize_text(c.text) for c in row.cells]
                if len(cells) < 2:
                    continue
                row_as_str: List[str] = []
                # Случай с хедером. Для удобсва добавляю колонку хедера к значению через ':' (Значение характеристики: Full HD)
                if has_header:
                    if j == 0:
                        continue
                    for head, val in zip(table_header, cells):
                        row_as_str.append("".join([head, ": ", val]))
                # Случай без хедера. Полагаю, первая колонка - ключ
                else:
                    key: str = cells[0]
                    val: str = " ".join(cells[1:])
                    row_as_str.append(f"{key}: {val}")

                row_as_str_joined: str = "; ".join(row_as_str)
                tables_kv[f"table_{i}"].append(row_as_str_joined)
        tables_lines = [val for key, val in tables_kv.items()]
        
        return np.concatenate(tables_lines)
    
parser = ContractParser(docs_paths[1])

paragraphs = parser.extract_clean_text()
tables = parser.extract_table_kv_from_docx()

In [45]:
from sentence_transformers import SentenceTransformer, util
# ----- Пример с одним пунктом из эталона -----

parser_contract = ContractParser(contract_path_WRONG)
paragraphs_contract = parser_contract.extract_clean_text(chunk_size = 40)
tables_contrac = parser_contract.extract_table_kv_from_docx()

parser_plan = PlanParser(plan_path_CORRECT)
anchor_vals = parser_plan.extract_table_kv_from_docx()
paragraphs_plan = parser_plan.extract_clean_text(chunk_size=40)

reference_chunks = [anchor_vals[11]]  # строки из плана-графика
document_chunks = paragraphs_contract # чанки по 40 слов
paragraphs_contract

['Контракт №',
 'на поставку компьютерной техники',
 'Идентификационный код закупки указывается в структурированном виде (цифровой форме) электронного контракта, сформированного с использованием единой информационной системы в сфере закупок.',
 'Государственное бюджетное учреждение Новосибирской области "Центр информационных технологий Новосибирской области" (сокращенное наименование - ГБУ НСО "ЦИТ НСО"), именуемый в дальнейшем "Заказчик" в лице руководителя учреждения Брумма Сергея Викторовича, действующего на основании Устава, с одной стороны, и _______________________,_______________________________(сокращенное наименование - __________)',
 'основании Устава, с одной стороны, и _______________________,_______________________________(сокращенное наименование - __________) именуем___ в дальнейшем "Поставщик", в лице ________________________, действующ___ на основании _______________________, с другой стороны, вместе именуемые в дальнейшем "Стороны", в соответствии с частью 12 статьи 9

In [8]:
#!pip install sbercloud
model = SentenceTransformer("sberbank-ai/sbert_large_nlu_ru")

In [30]:
# ----- Поиск чанков ближайших по смыслу к эталону -----
reference_embeddings = model.encode(reference_chunks, convert_to_tensor=True)
document_embeddings = model.encode(document_chunks, convert_to_tensor=True)
top_k = 15

closest_k = []
for i, ref_emb in enumerate(reference_embeddings):
    # Косинусное сходство
    cos_scores = util.cos_sim(ref_emb, document_embeddings)[0]
    # Берем top_k
    top_results = cos_scores.topk(k=top_k)
    
    print(f"\nПункт эталона: {reference_chunks[i]}")
    for score, idx in zip(top_results.values, top_results.indices):
        print(f"{score:.4f} -> {document_chunks[idx]}")
        closest_k.append(document_chunks[idx])


Пункт эталона: Сроки поставки товара, выполнения работ, оказания услуг по контракту: В течение 49 (сорока девяти) календарных дней с даты заключения Контракта.
0.9219 -> Срок поставки: в течение 49 (сорока девяти) рабочих дней с даты заключения Контракта.
0.9152 -> Поставщик не менее чем за 2 (два) рабочих дня до осуществления поставки Товара направляет в адрес Заказчика уведомление о времени и дате доставки Товара в место доставки.
0.9152 -> Поставщик не менее чем за 2 (два) рабочих дня до осуществления поставки Товара направляет в адрес Заказчика уведомление о времени и дате доставки Товара в место доставки.
0.9068 -> Поставщик получивший предложение об изменении условий Контракта, в течение 5 (пяти) рабочих дней со дня, следующего за днем получения предложения об изменении условий Контракта, по результатам рассмотрения такого предложения в порядке, установленном законодательством Российской Федерации о контрактной системе в сфере
0.9059 -> В течение гарантийного срока в случае возн

In [37]:
import requests
from shared_modules.llm_models import OPENAI_MODEL, get_chatGPT_client

client = get_chatGPT_client()
# Надо подумать над промптами

system_prompt = (
    "Ты — юридический помощник и эксперт по проверке документов. "
    "Твоя задача — проверять соответствие документов между эталонным пунктом и фрагментами другого документа. "
    "Обрати внимание на следующие моменты:\n"
    "1. Смысловая близость: даже если текст обрывчатый или переформулирован, ищи фрагменты, которые соответствуют по значению.\n"
    "2. Числовые данные: проверяй даты, суммы, коды, проценты, номера пунктов. Учитывай возможные ошибки записи (например, 21 вместо двадцать два).\n"
    "3. Словесные аналоги чисел дат и сумм: если числа написаны словами или смешаны с цифрами, тоже проверяй соответствие.\n"
    "4. Ошибки и несоответствия: выделяй все расхождения, указывая точный фрагмент, где они встречаются.\n"
    "5. Сохраняй структуру ответа: сначала перечисляй найденные совпадения, потом отмечай несоответствия с пояснениями."
)

user_prompt = (
    f"Эталонный пункт:\n{reference_chunks[0]}\n\n"
    f"Фрагмент документа:\n{'; '.join(closest_k)}\n\n"
    "Проанализируй внимательно и выдай отчет по соответствию, "
    "выделяя ключевые моменты и ошибки, в удобном для чтения виде."
    "То есть вначале напечатай эталонный пункт. А для соотвутвующих фрагментов с ошибкой предварительно напечатай сам фрагмент и после приводи анализ"
)

response = client.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}])

text_answer = response.choices[0].message.content
print(text_answer)

## Эталонный пункт:
**«Сроки поставки товара, выполнения работ, оказания услуг по контракту:  
В течение 49 (сорока девяти) календарных дней с даты заключения Контракта.»**

---

### Анализ и сопоставление фрагментов:

#### Соответствие сроков поставки товара:
- **Фрагмент:**  
  *«Срок поставки: в течение 49 (сорока девяти) рабочих дней с даты заключения Контракта.»*
  
  **Анализ:**  
  - Срок поставки указан верно, однако вместо календарного дня используется рабочий день. 
  - Фраза соответствует смыслу эталона, хотя формулировка отличается.

#### Несоответствия и замечания:
- **Фрагмент:**  
  *«срок: в течение 49 (сорока пяти) рабочих дней с даты заключения Контракта.»*  

  **Анализ:**  
  - Здесь указана неверная цифра — указано сорок пять рабочих дней вместо сорока девяти.  
  - Формулировка соответствует количеству дней, но количество указано неправильно.

---

Таким образом, существует несоответствие в числовом выражении срока поставки (указано 49 рабочих дней, тогда как долж

In [18]:

### НЕ РАБОТАЕТ [SSL: CERTIFICATE_VERIFY_FAILED]
client = get_chatGPT_client()
response = client.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": "Расскажи про себя"}])
text = response.choices[0].message.content
print(text)

Вот несколько ключевых моментов обо мне:

- **Русский язык — мой родной.** Общаюсь естественно и понятно, будто рядом сидит собеседник.
  
- **Работаю с текстом и информацией.** Могу помочь анализировать тексты, решать задачи, писать статьи, придумывать сценарии, учить и объяснять сложные темы простым языком.

- **Поддерживаю разные форматы.** Легко переключаюсь между устной речью (разговоры), логическими рассуждениями, математикой, программированием и креативностью.

- **Не занимаюсь предсказаниями реального времени.** Например, не смогу подсказать актуальные курсы валют или сегодняшнюю погоду.

- **Всегда стараюсь чётко обосновывать мысли.** Если задача требует анализа или решения, пошагово раскрываю детали.

Могу поддержать беседу, помочь разобраться в сложной теме или даже придумать забавный сюжет. Чем займёмся?
